In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)

In [2]:
use_cuda = True
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")
print(f"Using device: {device}")

CUDA available: True
Using device: cuda


In [3]:
# For loading the Diabetic Retinopathy Dataset

transforms_base = transforms.Compose([transforms.ToTensor()])

# full_dataset[i] is a a tuple of (image tensor, class index)
full_dataset = torchvision.datasets.DatasetFolder(root="./dataset", loader=torchvision.datasets.folder.default_loader, transform=transforms_base, extensions=[".png"])

In [4]:
print("Classes:", full_dataset.class_to_idx)
print("Total dataset size:", len(full_dataset))

healthy = 0
severe = 0
for data in full_dataset:
    if data[1] == 0:
        healthy += 1
    elif data[1] == 1:
        severe += 1
print("Total 'Healthy' samples:", healthy)
print("Total 'Severe' samples:", severe)

Classes: {'Healthy': 0, 'Severe DR': 1}
Total dataset size: 1190
Total 'Healthy' samples: 1000
Total 'Severe' samples: 190


In [5]:
'''
Training-Test-Validation Split
Train dataset: 80% of 'Healthy' + 80% of 'Severe'
Test dataset: 10% of 'Healthy' + 10% of 'Severe'
Validation dataset: 10% of 'Healthy' + 10% of 'Severe'
'''

targets = full_dataset.targets
train_indices = []
test_indices = []
validation_indices = []

for c in range(len(full_dataset.classes)):
    class_indices = [i for i, t in enumerate(targets) if t==c]
    torch.manual_seed(42)
    perm = torch.randperm(len(class_indices)).tolist()
    if c == 0:
        r = healthy
    else:
        r = severe
    a = int(r*0.8)
    b = int(r*0.9)
    train_indices.extend([class_indices[p] for p in perm[:a]])
    test_indices.extend([class_indices[p] for p in perm[a:b]])
    validation_indices.extend([class_indices[p] for p in perm[b:r]])

train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
test_dataset = torch.utils.data.Subset(full_dataset, test_indices)
validation_dataset = torch.utils.data.Subset(full_dataset, validation_indices)

In [6]:
# Dataloaders

batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False)
validation_dataloader = DataLoader(validation_dataset, batch_size, shuffle=False)